In [4]:
import pandas as pd

df = pd.read_csv("../data/raw-data/MEdata_07-03-26.csv")

df['WEEK'] = pd.to_datetime(df['WEEK'])
df = df.sort_values(['COUNTRY', 'WEEK'])
df.head()

,WEEK,REGION,COUNTRY,ADMIN1,EVENT_TYPE,SUB_EVENT_TYPE,EVENTS,FATALITIES,POPULATION_EXPOSURE,DISORDER_TYPE,ID,CENTROID_LATITUDE,CENTROID_LONGITUDE
20,2015-12-26,Middle East,Bahrain,Capital,Protests,Peaceful protest,3,0,38207.0,Demonstrations,285.0,26.1927,50.5508
612,2015-12-26,Middle East,Bahrain,Capital,Riots,Violent demonstration,3,0,91152.0,Demonstrations,285.0,26.1927,50.5508
1036,2015-12-26,Middle East,Bahrain,Muharraq,Protests,Peaceful protest,1,0,2866.0,Demonstrations,286.0,26.2547,50.6428
1346,2015-12-26,Middle East,Bahrain,Northern,Protests,Excessive force against protesters,2,0,18098.0,Political violence; Demonstrations,287.0,26.1468,50.4809
1356,2015-12-26,Middle East,Bahrain,Northern,Protests,Peaceful protest,2,0,16443.0,Demonstrations,287.0,26.1468,50.4809


In [6]:
#Target variable 
df['next_week_fatalities'] = df.groupby('COUNTRY')['FATALITIES'].shift(-1)

In [11]:
#Feature Engineering
df['fatalities_lag1'] = df.groupby('COUNTRY')['FATALITIES'].shift(1)
df['fatalities_lag2'] = df.groupby('COUNTRY')['FATALITIES'].shift(2)
df['fatalities_lag3'] = df.groupby('COUNTRY')['FATALITIES'].shift(3)

df['events_lag1'] = df.groupby('COUNTRY')['EVENTS'].shift(1)

df['fatalities_roll_mean'] = df.groupby('COUNTRY')['FATALITIES'] \
                               .rolling(4).mean().reset_index(0, drop=True)

df['fatalities_roll_std'] = df.groupby('COUNTRY')['FATALITIES'] \
                              .rolling(4).std().reset_index(0, drop=True)


In [12]:
df_model = df.dropna()
df.head()


,WEEK,REGION,COUNTRY,ADMIN1,EVENT_TYPE,SUB_EVENT_TYPE,EVENTS,FATALITIES,POPULATION_EXPOSURE,DISORDER_TYPE,ID,CENTROID_LATITUDE,CENTROID_LONGITUDE,next_week_fatalities,fatalities_lag1,fatalities_lag2,fatalities_lag3,events_lag1,fatalities_roll_mean,fatalities_roll_std
20,2015-12-26,Middle East,Bahrain,Capital,Protests,Peaceful protest,3,0,38207.0,Demonstrations,285.0,26.1927,50.5508,0.0,NaN,NaN,NaN,NaN,NaN,NaN
612,2015-12-26,Middle East,Bahrain,Capital,Riots,Violent demonstration,3,0,91152.0,Demonstrations,285.0,26.1927,50.5508,0.0,0.0,NaN,NaN,3.0,NaN,NaN
1036,2015-12-26,Middle East,Bahrain,Muharraq,Protests,Peaceful protest,1,0,2866.0,Demonstrations,286.0,26.2547,50.6428,0.0,0.0,0.0,NaN,3.0,NaN,NaN
1346,2015-12-26,Middle East,Bahrain,Northern,Protests,Excessive force against protesters,2,0,18098.0,Political violence; Demonstrations,287.0,26.1468,50.4809,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1356,2015-12-26,Middle East,Bahrain,Northern,Protests,Peaceful protest,2,0,16443.0,Demonstrations,287.0,26.1468,50.4809,0.0,0.0,0.0,0.0,2.0,0.0,0.0


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

features = [
    'fatalities_lag1','fatalities_lag2','fatalities_lag3',
    'events_lag1','fatalities_roll_mean','fatalities_roll_std'
]

X = df_model[features]
y = df_model['next_week_fatalities']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = RandomForestRegressor()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [14]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)
print("MAE:", mae)

MAE: 8.732367747014962


In [15]:
df_model['FATALITIES'].describe()

count    113635.000000
mean          4.753852
std          39.549020
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max        8102.000000
Name: FATALITIES, dtype: float64